# Simple unary binary-string tasks

Train the same local NCA architecture on either string reversal or bitwise NOT. These are string transformations, not integer operations: leading zeroes are data and are preserved. Training uses every binary string from length 1 through `TRAIN_MAX_LENGTH`; extrapolation uses only strings of exactly `EXTRAPOLATION_LENGTH`.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "run":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ncpu_computer import (
    ExperimentConfig, GeometryConfig, ModelConfig, TaskDataset, TrainingConfig,
    bitwise_not_task, evaluate, infer, load_model, reverse_task, train_seeds,
    validate_experiment,
)
from ncpu_computer.evaluation import format_results

In [ ]:
TASK_NAME = "reverse"  # "reverse" or "bit_not"
TRAIN_MAX_LENGTH = 4
EXTRAPOLATION_LENGTH = 8
TAIL_SLOTS = 2
HIDDEN_SIZE = 96
SEEDS = (0,)

UPDATES = 3000
BATCH_SIZE = 128
FREE_STEPS = 60
SUPERVISION_STEPS = 140
LEARNING_RATE = 2e-3
FINAL_LEARNING_RATE = 1e-4

In [ ]:
task_builders = {"reverse": reverse_task, "bit_not": bitwise_not_task}
if TASK_NAME not in task_builders:
    raise ValueError(f"unknown task: {TASK_NAME}")
task_builder = task_builders[TASK_NAME]

geometry = GeometryConfig(
    tape_slots=TRAIN_MAX_LENGTH + 1 + TAIL_SLOTS,
    stride=2,
    border_left=3,
    border_right=3,
    border_top=3,
    border_bottom=3,
)
model_config = ModelConfig(
    channels=5,
    hidden_size=HIDDEN_SIZE,
    fixed_kernels=("identity", "sobel_x", "sobel_y"),
    learnable_kernels=1,
    learnable_kernel_init="laplacian",
    gate="none",
    fire_rate=1.0,
    program_channel=0,
    io_channel=1,
    padding="zeros",
    max_abs_state=10.0,
)
training_config = TrainingConfig(
    updates=UPDATES,
    batch_size=BATCH_SIZE,
    free_steps=FREE_STEPS,
    supervision_steps=SUPERVISION_STEPS,
    learning_rate=LEARNING_RATE,
    final_learning_rate=FINAL_LEARNING_RATE,
    weight_decay=2e-5,
    grad_clip=0.8,
    terminator_weight=0.0,
    tail_weight=0.0,
    validation_every=50,
    checkpoint_every=50,
    device="auto",
)
config = ExperimentConfig(geometry, model_config, training_config)
train_task = task_builder(TRAIN_MAX_LENGTH, include_shorter=True)
train_data = TaskDataset.from_task(train_task, geometry.tape_slots)
checkpoint_dir = (
    ROOT / "checkpoints" / f"{TASK_NAME}_train{TRAIN_MAX_LENGTH}"
)
checkpoint_path = checkpoint_dir / "best.pt"

print(validate_experiment(config, train_data))
print(f"task: {train_task.name}; examples: {len(train_data)}")
for example in train_task.examples[:8]:
    print(f"{example.input:>{TRAIN_MAX_LENGTH}} -> {example.target}")

In [ ]:
RUN_TRAINING = False
RESUME = False

if RUN_TRAINING:
    seed_results = train_seeds(
        config,
        train_data,
        seeds=SEEDS,
        checkpoint_dir=checkpoint_dir,
        resume=RESUME,
    )
    seed_results

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path)
    if checkpoint["dataset_signature"] != train_data.signature:
        raise ValueError("checkpoint dataset differs from this notebook")
    trained_schedule = trained_config.training

    train_result = evaluate(
        model,
        trained_config.geometry,
        train_data,
        steps=trained_schedule.rollout_steps,
        step_start=trained_schedule.supervision_start,
        step_end=trained_schedule.supervision_end,
        batch_size=BATCH_SIZE,
    )

    extrapolation_geometry = replace(
        trained_config.geometry,
        tape_slots=EXTRAPOLATION_LENGTH + 1 + TAIL_SLOTS,
    )
    extrapolation_task = task_builder(
        EXTRAPOLATION_LENGTH, include_shorter=False
    )
    extrapolation_data = TaskDataset.from_task(
        extrapolation_task, extrapolation_geometry.tape_slots
    )
    extrapolation_result = evaluate(
        model,
        extrapolation_geometry,
        extrapolation_data,
        steps=trained_schedule.rollout_steps,
        step_start=trained_schedule.supervision_start,
        step_end=trained_schedule.supervision_end,
        batch_size=BATCH_SIZE,
    )
    print(format_results([train_result, extrapolation_result]))

In [ ]:
RUN_INFERENCE = False
SAMPLE_INPUT = "01001101"

if RUN_INFERENCE:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path)
    inference_geometry = replace(
        trained_config.geometry,
        tape_slots=len(SAMPLE_INPUT) + 1 + TAIL_SLOTS,
    )
    sample_task = task_builder(len(SAMPLE_INPUT), include_shorter=False)
    expected = next(
        example.target
        for example in sample_task.examples
        if example.input == SAMPLE_INPUT
    )
    prediction = infer(
        model,
        inference_geometry,
        SAMPLE_INPUT,
        steps=trained_config.training.rollout_steps,
    )
    predicted_string = (
        prediction.interpreted.binary_strings[0]
        if prediction.interpreted.valid
        else None
    )
    print(f"continuous tape: {prediction.values.tolist()}")
    print(f"raw ternary:    {prediction.interpreted.raw}")
    print(f"prediction:     {predicted_string}")
    print(f"expected:       {expected}")